In [6]:
import pandas as pd
import os

In [7]:
result_files = os.listdir("../../docs/results")

dfs = []
for file in result_files:
    if file.endswith('.csv'):
        file_path = os.path.join("../../docs/results", file)
        dfs.append(pd.read_csv(file_path))

df = pd.concat(dfs, ignore_index=True)

In [8]:
df.head()

,Model,Approach,Features,Fold,MSE,RMSE,MAE,R2,SMAPE
0,RNN,Uni,noExog,Fold1,90146.77,300.24,123.29,0.8384,68.94%
1,RNN,Uni,noExog,Fold2,72547.65,269.35,116.39,0.8553,68.59%
2,RNN,Uni,noExog,Fold3,68480.28,261.69,106.51,0.8696,64.13%
3,RNN,Uni,noExog,Average,77058.23 ± 11088.68,277.10 ± 19.82,115.40 ± 8.50,0.8544 ± 0.0156,67.22% ± 2.51%
4,RNN,Uni,Exog,Fold1,128578.29,358.58,147.06,0.7695,81.30%


In [9]:
# Create a list to store the modified dataframes
modified_rows = []

# Iterate through the dataframe
for idx, row in df.iterrows():
    if row['Fold'] == 'Average':
        # Create the Average row without std
        avg_row = row.copy()
        std_row = row.copy()
        std_row['Fold'] = 'Standard Deviation'
        
        # Process each metric column
        for col in ['MSE', 'RMSE', 'MAE', 'R2']:
            if '±' in str(row[col]):
                parts = str(row[col]).split('±')
                avg_row[col] = parts[0].strip()
                std_row[col] = parts[1].strip()
            else:
                std_row[col] = ''
        
        # Handle SMAPE column if it exists
        if 'SMAPE' in row.index and '±' in str(row['SMAPE']):
            parts = str(row['SMAPE']).split('±')
            avg_row['SMAPE'] = parts[0].strip()
            std_row['SMAPE'] = parts[1].strip()
        
        modified_rows.append(avg_row)
        modified_rows.append(std_row)
    else:
        modified_rows.append(row)

# Create the new dataframe
df = pd.DataFrame(modified_rows).reset_index(drop=True)

# Remove percentage signs from SMAPE column if it exists
df['SMAPE'] = df['SMAPE'].astype(str).str.replace('%', '').astype(float)

In [10]:
df.head()

,Model,Approach,Features,Fold,MSE,RMSE,MAE,R2,SMAPE
0,RNN,Uni,noExog,Fold1,90146.77,300.24,123.29,0.8384,68.94
1,RNN,Uni,noExog,Fold2,72547.65,269.35,116.39,0.8553,68.59
2,RNN,Uni,noExog,Fold3,68480.28,261.69,106.51,0.8696,64.13
3,RNN,Uni,noExog,Average,77058.23,277.10,115.40,0.8544,67.22
4,RNN,Uni,noExog,Standard Deviation,11088.68,19.82,8.50,0.0156,2.51


In [11]:
df.to_parquet("../../docs/results/all_results.parquet", index=False)